# Reimbursement Agent - Experimental Notebook

This notebook deconstructs the `ReimbursementPolicyAgent` into individual cells for experimentation.

## What's Exposed:
- ✅ All prompts (editable)
- ✅ Complete 5-node pipeline
- ✅ Full LLM interaction visibility
- ✅ Pydantic validation at each stage
- ✅ Dynamic table structure generation
- ✅ Keyword detection for API search
- ✅ High reasoning LLM for better extraction

## Latest Updates:
1. **Keyword Detection**: LLM detects relevant keywords from codes (e.g., "99291,99292" → "critical care")
2. **High Reasoning LLM**: Uses "high" reasoning effort for better extraction quality
3. **Company Filtering**: Filters policies by approved company domains
4. **Progress Indicators**: Added [KEYWORD], [API], [SNOWFLAKE], [FILTER], [DEDUP] tags

## Workflow:
1. Setup environment and configuration
2. Input CPT codes
3. **Node 1**: Keyword detection + Search policies (API + filtering)
4. **Node 2**: Fetch policy content (Snowflake)
5. **Node 3**: Extract rules with LLM (high reasoning)
6. **Node 4**: Analyze table structure
7. **Node 5**: Format output
8. Export and analyze results

## Section 1: Setup & Configuration

In [1]:
### 1.1 Imports

import sys
import json
import re
import time
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional
from datetime import datetime
from collections import defaultdict

# Add packages to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "packages" / "agents" / "src"))
sys.path.insert(0, str(project_root / "packages" / "core" / "src"))
sys.path.insert(0, str(project_root / "packages" / "utils" / "src"))

print(f"Project root: {project_root}")
print("✓ Imports successful")

In [2]:
# Import Pydantic models
from deep_research_agents.models.reimbursement_models import (
    PolicyExtractionResponse,
    ColumnLabelsResponse,
    RuleSummary
)

print("✓ Agent modules imported")

2026-05-27 14:19:55,664 - policy_extractor.system - INFO - === Policy Extractor Logging Initialized ===
2026-05-27 14:19:55,665 - policy_extractor.system - INFO - Log directory: c:\projects\coc\idiscovery-deep-research\notebooks\prototyping\logs
2026-05-27 14:19:55,666 - policy_extractor.system - INFO - Max file size: 50.0MB
2026-05-27 14:19:55,667 - policy_extractor.system - INFO - Backup count: 10
2026-05-27 14:19:55,667 - policy_extractor.system - INFO - Console output enabled: True
2026-05-27 14:19:55,668 - policy_extractor.system - INFO - Console log level: INFO
2026-05-27 14:19:55,669 - policy_extractor.system - INFO - Console stream: stdout
2026-05-27 14:19:55,669 - policy_extractor.system - INFO - Process ID: 29444
2026-05-27 14:19:55,670 - policy_extractor.system - INFO - Component log levels:
2026-05-27 14:19:55,671 - policy_extractor.system - INFO -   policy_extractor.snowflake_store: WARNING
2026-05-27 14:19:55,671 - policy_extractor.system - INFO -   policy_extractor.snowf

### 1.2 Configuration (Editable)

In [3]:
# ============= CONFIGURATION (EDIT HERE) =============

# LLM settings
LLM_MODEL = "gpt-5.4"
LLM_TEMPERATURE = 0.0  # Use 0 for extraction tasks
LLM_REASONING_EFFORT = "high"  # Use high reasoning for better extraction quality

# API settings
CARELON_API_URL = "https://policy-comparison-api.carelon.com/policy_comparison/search"

# Retry settings
RETRY_DELAY = 30  # seconds
MAX_RETRIES = 1

# Keyword detection settings
USE_KEYWORD_DETECTION = True  # Use LLM to detect search keywords from codes

# Debug settings
DEBUG_MODE = True
SHOW_FULL_PROMPTS = True
SHOW_RAW_RESPONSES = True

print("Configuration:")
print(f"  LLM Model: {LLM_MODEL}")
print(f"  Temperature: {LLM_TEMPERATURE}")
print(f"  LLM Reasoning: {LLM_REASONING_EFFORT}")
print(f"  Keyword Detection: {USE_KEYWORD_DETECTION}")
print(f"  Debug Mode: {DEBUG_MODE}")

Configuration:
  LLM Model: gpt-5.4
  Temperature: 0.0
  LLM Reasoning: high
  Keyword Detection: True
  Debug Mode: True


### 1.3 Initialize CredentialProvider

**Note:** LLM is initialized internally by the agent via `AgentBase`. No need to create LLM separately.

In [4]:
# Initialize CredentialProvider (used by Snowflake and Agent)
from deep_research_core.base_agent import CredentialProvider

creds = CredentialProvider.get_instance()

print("✓ CredentialProvider initialized")
print()
print("Note: LLM will be initialized by the agent itself via AgentBase.")

✓ CredentialProvider initialized

Note: LLM will be initialized by the agent itself via AgentBase.


### 1.4 Initialize Snowflake

In [5]:
# Initialize Snowflake helper
from deep_research_utils.snowflake_helper import SnowparkHelper

snowflake_creds = creds.get_snowflake_credentials()
snowflake_helper = SnowparkHelper(
    connection_type="programmatic",
    batch_size=10000,
    max_workers=6,
    enable_metrics=True,
    connection_pool_size=4,
    **snowflake_creds
)
print("✓ Snowflake helper initialized")

[CredentialProvider] Snowflake connection type: programmatic
[CredentialProvider] Using programmatic credentials for Snowflake
[CredentialProvider] Password length: 231 characters
2026-05-27 14:19:56,614 - deep_research_utils.snowflake_helper - INFO - 🔑 Configuring PROGRAMMATIC connection (password-based authentication)
2026-05-27 14:20:00,025 - deep_research_utils.snowflake_helper - INFO - Snowflake session created successfully
2026-05-27 14:20:09,614 - deep_research_utils.snowflake_helper - INFO - Initialized connection pool with 3 additional sessions
✓ Snowflake helper initialized


## Section 2: Input Data

In [6]:
# ============== Input from pattern json ===================== 
import json

pattern_json = "pattern_results_20260520_IP_AUTH-Commercial-202604-R3-202601.json"
codes_inputs = {}

with open(pattern_json, 'r') as file:
    pattern_data = json.load(file)

patterns = pattern_data['output']['business_patterns']
cards = pattern_data['output']['cards']

card_id_map = {
    card['card_id']: card for card in cards
}

for i, pattern in enumerate(patterns):
    source_card_ids = pattern.get("source_card_ids", [])

    for card_id in source_card_ids:
        card = card_id_map.get(card_id, {})
        filters = card.get("filters", [])

        for filter in filters:
            if filter.get("field", "")=="drg_name":
                codes_inputs[i] = codes_inputs.get(i, []) + [filter.get("value", "")]

codes_inputs


{4: ['ECMO or Tracheostomy with Mechanical Ventilation >96 Hours or Principal Diagnosis Except Face, Mouth and Neck with Major O.R. Procedures',
  'Heart Transplant or Implant of Heart Assist System with MCC',
  'Tracheostomy with Mechanical Ventilation >96 Hours or Principal Diagnosis Except Face, Mouth and Neck without Major O.R. Procedures',
  'Major Small and Large Bowel Procedures with MCC'],
 7: ['Alcohol, Drug Abuse or Dependence without Rehabilitation Therapy without MCC',
  'Alcohol, Drug Abuse or Dependence without Rehabilitation Therapy without MCC']}

### 2.1 CPT Code Input (Editable)

In [7]:
import json

# ['Cesarean Section without Sterilization without CC/MCC',
#   'Vaginal Delivery without Sterilization/D&C with CC',
#   'Vaginal Delivery Without Sterilization or D&C without CC/MCC']

codes = json.dumps(['Cesarean Section without Sterilization without CC/MCC',
   'Vaginal Delivery without Sterilization/D&C with CC',
   'Vaginal Delivery Without Sterilization or D&C without CC/MCC'])
print(codes)

["Cesarean Section without Sterilization without CC/MCC", "Vaginal Delivery without Sterilization/D&C with CC", "Vaginal Delivery Without Sterilization or D&C without CC/MCC"]


In [8]:
# ============= CPT CODES (EDIT HERE) =============

# cpt_codes = "Alcohol, Drug Abuse or Dependence without Rehabilitation Therapy without MCC;Alcohol, Drug Abuse or Dependence without Rehabilitation Therapy without MCC"
cpt_codes = codes

print(f"CPT Codes: {cpt_codes}")
print(f"Number of codes: {len([c.strip() for c in cpt_codes.split(';') if c.strip()])}")

CPT Codes: ["Cesarean Section without Sterilization without CC/MCC", "Vaginal Delivery without Sterilization/D&C with CC", "Vaginal Delivery Without Sterilization or D&C without CC/MCC"]
Number of codes: 1


### 2.2 Validate Input

In [9]:
# Validate CPT codes format
codes_list = [c.strip() for c in cpt_codes.split(';') if c.strip()]

if not codes_list:
    print("❌ No valid CPT codes provided")
else:
    print(f"✓ Input validated: {len(codes_list)} code(s)")
    for code in codes_list:
        print(f"  - {code}")

✓ Input validated: 1 code(s)
  - ["Cesarean Section without Sterilization without CC/MCC", "Vaginal Delivery without Sterilization/D&C with CC", "Vaginal Delivery Without Sterilization or D&C without CC/MCC"]


## Section 3: Node 1 - Policy Search (with Keyword Detection)

### 3.1 Keyword Detection (NEW FEATURE)

In [10]:
# Detect search keyword from codes (NEW FEATURE)
print("[KEYWORD DETECTION] Detecting search keyword from codes...")
print("="*80)

if USE_KEYWORD_DETECTION:
    # Create temporary agent for LLM access
    from deep_research_core.base_agent import AgentBase
    
    class TempAgent(AgentBase):
        def __init__(self):
            super().__init__(
                agent_name="temp_keyword_detector",
                state_class=dict,
                llm_reasoning_effort="high"
            )
        
        @property
        def node_name(self):
            return "keyword_detection"
        
        def node_function(self, state):
            return state
        
        def extract_result(self, final_state):
            return final_state
    
    print("[LLM] Initializing temporary LLM client...")
    temp_agent = TempAgent()
    llm = temp_agent.llm
    print("[LLM] ✓ LLM client initialized")
    
    prompt = f"""Given the following medical codes: {cpt_codes}

Identify the most relevant keywords or short phrases (1-3 words) that best describes what these codes relate to for searching reimbursement policies.
The input could be CPT codes/descriptions, DRG codes/descriptions, ICD codes/descriptions, or HCPCS codes/descriptions.
If there are multiple of them, they should be comma seperated.
The keywords should be specific enough to find relevant policies but general enough to cover related codes.
If there are common terms in the codes, they should appear in the output.
The keywords should either come from code descriptions if the descriptions are provided.
If the code descriptions are not privided, then the keywords should be inferred from the codes.
Do not use special characters in the answer, it should only be words and phrases.

Return ONLY the keyword or short phrase, nothing else."""

    messages = [
        {"role": "system", "content": "You are a medical coding expert. Generate concise search keywords."},
        {"role": "user", "content": prompt}
    ]
    
    if SHOW_FULL_PROMPTS:
        print("\\n[PROMPT] System Message:")
        print("-"*80)
        print(messages[0]["content"])
        print("-"*80)
        print("\\n[PROMPT] User Message:")
        print("-"*80)
        print(messages[1]["content"])
        print("-"*80)
    
    try:
        print("\\n[LLM] Invoking LLM for keyword detection...")
        response = llm.invoke(messages)
        
        if SHOW_RAW_RESPONSES:
            print("\\n[RESPONSE] LLM Response:")
            print("-"*80)
            print(response.content)
            print("-"*80)
        
        search_keyword = response.content.strip().strip('"').strip("'")
        keyword_source = "llm"
        print(f"\\n[KEYWORD DETECTION] ✓ LLM detected: '{search_keyword}'")
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f"\\n[KEYWORD DETECTION] ⚠ LLM failed: {e}, using first code")
        search_keyword = codes_list[0]
        keyword_source = "fallback"
else:
    search_keyword = codes_list[0]
    keyword_source = "direct"
    print(f"\\n[KEYWORD DETECTION] Using first code: '{search_keyword}'")

print(f"\\n[KEYWORD DETECTION] Search keyword: '{search_keyword}' (source: {keyword_source})")
print("="*80)

[KEYWORD DETECTION] Detecting search keyword from codes...
[LLM] Initializing temporary LLM client...
2026-05-27 14:20:10,094 - deep_research_utils.ehap - INFO - Requesting new access token from https://api.horizon.elevancehealth.com/v2/oauth2/token with client_id: iYfcOHQHPTaAn3BckTey49nJg93QsBmE


c:\projects\coc\idiscovery-deep-research\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.horizon.elevancehealth.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


2026-05-27 14:20:11,044 - deep_research_utils.ehap - INFO - Access token generated successfully.
**TOKEN** **TOKEN** **TOKEN** 
[LLM] ✓ LLM client initialized
\n[PROMPT] System Message:
--------------------------------------------------------------------------------
You are a medical coding expert. Generate concise search keywords.
--------------------------------------------------------------------------------
\n[PROMPT] User Message:
--------------------------------------------------------------------------------
Given the following medical codes: ["Cesarean Section without Sterilization without CC/MCC", "Vaginal Delivery without Sterilization/D&C with CC", "Vaginal Delivery Without Sterilization or D&C without CC/MCC"]

Identify the most relevant keywords or short phrases (1-3 words) that best describes what these codes relate to for searching reimbursement policies.
The input could be CPT codes/descriptions, DRG codes/descriptions, ICD codes/descriptions, or HCPCS codes/description

### 3.2 API Call with Detected Keyword

In [40]:
# Search for policies using detected keyword
import requests
from deep_research_utils.app_constant import AppConstants

search_keyword = "obstetric"
print(f"[API] Calling Carelon API with keyword: {search_keyword}")
url = f"{CARELON_API_URL}?keyword={search_keyword}&user_type=onshore"
print(f"[API] URL: {url}")

response = requests.post(url, headers={}, data={}, verify=AppConstants.SSL_CERT_FILE)
response.raise_for_status()
raw_data = response.json()

print(f"[API] ✓ Found {len(raw_data.get('sentencelist', []))} result(s) from API")

[API] Calling Carelon API with keyword: obstetric
[API] URL: https://policy-comparison-api.carelon.com/policy_comparison/search?keyword=obstetric&user_type=onshore
[API] ✓ Found 1107 result(s) from API


### 3.3 Filter by Approved Companies

In [12]:
# Define approved companies and their domains
APPROVED_COMPANIES = [
    {"name": "Elevance Health", "domains": ["anthem.com", "anthembluecross.com", "providers.anthem.com"]},
    {"name": "Molina Medicaid", "domains": ["molinahealthcare.com", "molinamarketplace.com"]},
    {"name": "Premera Blue Cross", "domains": ["premera.com"]},
    {"name": "Kaiser Permanente WA", "domains": ["wa-provider.kaiserpermanente.org"]},
    {"name": "Highmark", "domains": ["providers.highmark.com", "highmarkprc.com", "highmarkhealthoptions.com", "highmarkblueshield.com"]},
    {"name": "BCBS Michigan", "domains": ["bcbsm.com"]},
    {"name": "Regence", "domains": ["regence.com"]},
    {"name": "BCBS Nebraska", "domains": ["nebraskablue.com", "medicalpolicy.nebraskablue.com"]},
    {"name": "Cigna", "domains": ["cigna.com", "static.cigna.com", "providernewsroom.com"]},
    {"name": "United Health", "domains": ["uhcprovider.com", "www.uhcprovider.com"]}
]

approved_domains = []
for company in APPROVED_COMPANIES:
    approved_domains.extend(company["domains"])

print(f"[FILTER] Approved companies: {len(APPROVED_COMPANIES)}")
print(f"[FILTER] Total approved domains: {len(approved_domains)}")

[FILTER] Approved companies: 10
[FILTER] Total approved domains: 20


In [13]:
# Filter policies by approved company domains
from urllib.parse import urlparse

def is_approved_url(url):
    if not url:
        return False
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        for approved_domain in approved_domains:
            if approved_domain.lower() in domain:
                return True
        return False
    except:
        return False

print(f"[FILTER] Filtering by approved company domains...")
sentencelist = raw_data.get('sentencelist', [])
filtered_sentences = []

for policy in sentencelist:
    if isinstance(policy, dict):
        external_link = policy.get('external_link', '')
        policy_link = policy.get('policy_link', '')
        if is_approved_url(external_link) or is_approved_url(policy_link):
            filtered_sentences.append(policy)

raw_data['sentencelist'] = filtered_sentences
print(f"[FILTER] Policies before filter: {len(sentencelist)}")
print(f"[FILTER] Policies after filter: {len(filtered_sentences)}")
print(f"[FILTER] ✓ Company filter applied")

[FILTER] Filtering by approved company domains...
[FILTER] Policies before filter: 660
[FILTER] Policies after filter: 217
[FILTER] ✓ Company filter applied


In [14]:
import pandas as pd

# Convert to DataFrame and filter
print(f"[FILTER] Converting API response to DataFrame...")
df = pd.DataFrame(raw_data)
df = df['sentencelist'].apply(pd.Series)

# Filter by file type (PDF only)
if 'policy_link' in df.columns:
    df["file_type"] = df.policy_link.str.split(".").str[-1]
    df = df[df['file_type'] == 'pdf']
    print(f"[FILTER] After PDF filter: {len(df)} policies")

# Filter by policy type (Reimbursement only)
if 'policy_type' in df.columns:
    df = df[df['policy_type'] == 'Reimbursement']
    print(f"[FILTER] After Reimbursement filter: {len(df)} policies")

# Remove duplicates
if 'payor' in df.columns and 'external_link' in df.columns:
    df = df.drop_duplicates(subset=['payor', 'external_link'])
    print(f"[DEDUP] After deduplication: {len(df)} policies")

print(f"[FILTER] ✓ Filtering complete")

[FILTER] Converting API response to DataFrame...
[FILTER] After PDF filter: 217 policies
[FILTER] After Reimbursement filter: 144 policies
[DEDUP] After deduplication: 32 policies
[FILTER] ✓ Filtering complete


In [15]:
df.head()

,policy_id,page_num,payor,policy_title,policy_link,external_link,policy_type,lob,claim_type,state,context,keyword,active_status,product_class,policy_score,update_date,file_type
1,RP_COM_CIG_00037,"2, 3, 4",Cigna,Multiple Births,s3://eai-aifs-cai-coc-pr-nogbd-s3-bucket-dataz...,https://static.cigna.com/assets/chcp/secure/pd...,Reimbursement,Commercial,CMS-1500,"AZ, CA, CO, CT, FL, GA, IL, IN, MD, MO, NJ, NC...",[{'sentence': 'Cigna does not typically allow ...,"cesarean section,vaginal delivery",Active,N/A,85,2026-02-17,pdf
2,RP_COM_PBC_WA_00049,"2, 4, 5, 7, 10",Premera Blue Cross,Maternity Services,s3://eai-aifs-cai-coc-pr-nogbd-s3-bucket-dataz...,https://www.premera.com/paymentpolicies/cmi_12...,Reimbursement,Commercial,CMS-1500,WA,[{'sentence': '• Inpatient post-delivery recov...,"cesarean section,vaginal delivery",Active,N/A,70,2025-10-07,pdf
3,RP_COM_PBC_WA_00046,"1, 3",Premera Blue Cross,Multiple Deliveries/Births,s3://eai-aifs-cai-coc-pr-nogbd-s3-bucket-dataz...,https://www.premera.com/paymentpolicies/cmi_05...,Reimbursement,Commercial,CMS-1500,WA,[{'sentence': 'Policy The Plan reimburses mult...,"cesarean section,vaginal delivery",Active,N/A,70,2026-03-12,pdf
5,RP_COM_CIG_00021,"3, 4, 7, 8",Cigna,Global Maternity/Obstetric Package,s3://eai-aifs-cai-coc-pr-nogbd-s3-bucket-dataz...,https://static.cigna.com/assets/chcp/secure/pd...,Reimbursement,Commercial,CMS-1500,"AZ, CA, CO, CT, FL, GA, IL, IN, MD, MO, NJ, NC...",[{'sentence': 'Coding/Billing Information Code...,vaginal delivery,Active,N/A,58,2026-02-23,pdf
6,RP_GBD_UHC_00214,"2, 5, 6, 7, 8, 10",United Health,"Obstetrical Services Policy, Professional for ...",s3://eai-aifs-cai-coc-pr-nogbd-s3-bucket-dataz...,https://www.uhcprovider.com/content/dam/provid...,Reimbursement,Medicaid,CMS-1500,LA,[{'sentence': '24 hours of delivery • Manageme...,"cesarean section,vaginal delivery",Active,N/A,58,2024-02-02,pdf


In [16]:
df.to_csv('ob_policies.csv')

## Section 4: Node 2 - Fetch Policy Content (Snowflake)

In [17]:
### 4.1 Fetch Policy Hashes from Snowflake

In [18]:
# Get policy hashes from Snowflake
policy_ids = df.policy_id.unique().tolist()

sql = """
SELECT PLCY_ID, PDF_HASH_VAL_ID
FROM P01_COC.COC_DTI_NOGBD.PLCY_MTDTA
WHERE PLCY_ID IN ({policy_ids})
  AND ACTV_STTS_NM = 'Active'
  AND STTS_NM NOT ILIKE '%delete%'
QUALIFY ROW_NUMBER() OVER (PARTITION BY PLCY_ID ORDER BY RCRD_DT DESC) = 1
"""
policy_ids_str = ", ".join([f"'{pid}'" for pid in policy_ids])
df_hash = snowflake_helper.execute_query_and_return_pandas_df(sql.format(policy_ids=policy_ids_str))

print(f"[SNOWFLAKE] Retrieved hashes for {len(df_hash)} policies")

# Merge and deduplicate by hash
df = df.merge(df_hash, left_on='policy_id', right_on='PLCY_ID', how='left')
df = df.drop_duplicates(subset=['payor', 'PDF_HASH_VAL_ID'])

print(f"[DEDUP] ✓ After hash deduplication: {len(df)} unique policies")

[SNOWFLAKE] Retrieved hashes for 32 policies
[DEDUP] ✓ After hash deduplication: 32 unique policies


In [19]:
# Fetch full policy content from Snowflake
policy_ids = df.policy_id.unique().tolist()

print(f"[SNOWFLAKE] Fetching content for {len(policy_ids)} policies...")

sql = """
SELECT PLCY_ID, PAYOR_NM, PLCY_URL_TXT,
       LISTAGG(PAGE_DATA_TXT, '\\n') WITHIN GROUP (ORDER BY PAGE_NBR) AS POLICY_TEXT
FROM P01_COC.COC_DTI_ETL_SEMANTIC_NOGBD.PLCY_MTDTA_SRCH_DTL 
WHERE PLCY_ID IN ({policy_ids})
GROUP BY 1, 2, 3
"""
policy_ids_str = ", ".join([f"'{pid}'" for pid in policy_ids])
df_content = snowflake_helper.execute_query_and_return_pandas_df(sql.format(policy_ids=policy_ids_str))

policy_content = df_content.to_dict('records')

print(f"[SNOWFLAKE] ✓ Retrieved content for {len(policy_content)} policies")

# Display sample
if policy_content:
    first_policy = policy_content[0]
    print(f"\\n[SAMPLE] {first_policy['PLCY_ID'][:50]}...")
    print(f"[LENGTH] {len(first_policy['POLICY_TEXT'])} characters")

[SNOWFLAKE] Fetching content for 32 policies...
[SNOWFLAKE] ✓ Retrieved content for 32 policies
\n[SAMPLE] RP_GBD_UHC_00214...
[LENGTH] 37355 characters


## Section 5: Node 3 - Rule Extraction (LLM)

### 5.1 Extraction Prompts (Editable)

In [20]:
# ============= PROMPTS (EDIT HERE) =============

SYSTEM_PROMPT = """You are an expert at analyzing insurance policies and extracting key information.
Read the policy text and return valid JSON only.
Follow the exact schema provided.
Do not output markdown or explanatory text.
Use null or [] when information is missing."""

USER_PROMPT_TEMPLATE = """Codes: {cpt_codes}

Policy Text: {policy_text}

Extract adjudication rules for the requested codes. Return ONLY valid JSON matching this schema:

{{
  "policy_metadata": {{
    "policy_title": "string",
    "effective_date": "MM/DD/YYYY format or if it is another format, convert to this format",
    "payer_category": "Medicaid/Commercial/Medicare Advantage",
    "appeals_process_documented": boolean
  }},
  "results": [{{
    "code": "CPT code",
    "site_of_service": "Where code is allowed/restricted",
    "bundling_logic": "Bundling rules",
    "code_interactions": "How code interacts with others",
    "modifier_usage": "Modifier requirements",
    "denial_conditions": "Common denial reasons",
    "unit_pricing_logic": "Time/unit rules",
    "documentation_requirements": "Documentation needed",
    "evidence_summary": "Brief summary (MAX 15 words)"
  }}]
}}"""

print("✓ Prompts loaded")

✓ Prompts loaded


### 5.2 Extract Rules from Policies

In [21]:
import json
# Extract rules from each policy
results = []
total = len(policy_content)

print(f"[LLM] Extracting rules from {total} policies...")
print("="*80)

for i, policy in enumerate(policy_content):
    policy_id = policy['PLCY_ID']
    policy_text = policy['POLICY_TEXT']
    
    print(f"\\n[{i+1}/{total}] Processing: {policy_id}")
    
    # Build prompt
    user_prompt = USER_PROMPT_TEMPLATE.format(
        policy_text=policy_text,
        cpt_codes=cpt_codes
    )
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    
    try:
        # Invoke LLM
        response = llm.invoke(messages)
        content = response.content.strip()
        
        # Remove markdown code fences if present
        if content.startswith("```"):
            lines = content.split("\\n")
            if len(lines) > 2:
                content = "\\n".join(lines[1:-1])
        
        # Parse JSON
        data = json.loads(content)
        
        # Validate with Pydantic
        validated = PolicyExtractionResponse(**data)
        
        # Add policy ID
        result = validated.model_dump()
        result["PLCY_ID"] = policy_id
        result['PAYOR_NM'] = policy['PAYOR_NM']
        result['PLCY_URL'] = policy['PLCY_URL_TXT']
        results.append(result)
        
        print(f"  ✓ Extracted {len(result.get('results', []))} rule(s)")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        results.append(None)

successful = sum(1 for r in results if r is not None)
print(f"\\n{'='*80}")
print(f"[LLM] ✓ Successfully processed {successful}/{total} policies")

[LLM] Extracting rules from 32 policies...
\n[1/32] Processing: RP_GBD_UHC_00214
  ✓ Extracted 3 rule(s)
\n[2/32] Processing: RP_GBD_UHC_00177
  ✓ Extracted 3 rule(s)
\n[3/32] Processing: RP_COM_UHC_00071
  ✓ Extracted 3 rule(s)
\n[4/32] Processing: RP_COM_PBC_WA_00088
  ✓ Extracted 3 rule(s)
\n[5/32] Processing: RP_COM_CIG_00031
  ✓ Extracted 3 rule(s)
\n[6/32] Processing: RP_COM_PBC_WA_00017
  ✓ Extracted 3 rule(s)
\n[7/32] Processing: RP_COM_HIGHMARK_BCBS_DE_PA_NY_WV_28194710002
  ✓ Extracted 3 rule(s)
\n[8/32] Processing: RP_GBD_MOH_00173
  ✓ Extracted 3 rule(s)
\n[9/32] Processing: RP_GBD_ELV_EXTERNAL_00380
  ✓ Extracted 3 rule(s)
\n[10/32] Processing: RP_GBD_UHC_00336
  ✓ Extracted 3 rule(s)
\n[11/32] Processing: RP_COM_PBC_WA_00046
  ✓ Extracted 3 rule(s)
\n[12/32] Processing: RP_GBD_UHC_00573
  ✓ Extracted 3 rule(s)
\n[13/32] Processing: RP_COM_CIG_00037
  ✓ Extracted 3 rule(s)
\n[14/32] Processing: RP_COM_KSPN_00024
  ✓ Extracted 3 rule(s)
\n[15/32] Processing: RP_COM_CIG_0004

## Section 6: Export Results

In [22]:
# Export results to JSON
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"reimbursement_agent_results_{timestamp}.json"

export_data = {
    "cpt_codes": cpt_codes,
    "search_keyword": search_keyword,
    "keyword_source": keyword_source,
    "total_policies_processed": len(results),
    "successful_extractions": successful,
    "results": [r for r in results if r is not None]
}

with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"[EXPORT] ✓ Results saved to: {output_file}")
print(f"[EXPORT]   Policies: {successful}")
# print(f"[EXPORT]   File size: {Path(output_file).stat().st_size:,} bytes")

[EXPORT] ✓ Results saved to: reimbursement_agent_results_20260527_150330.json
[EXPORT]   Policies: 32


## Summary

**Reimbursement Agent Experimental Notebook - Complete Workflow**

This notebook successfully deconstructs the ReimbursementAgent pipeline:

✅ **Section 1**: Environment setup and configuration  
✅ **Section 2**: CPT code input and validation  
✅ **Section 3**: Policy search with LLM keyword detection  
✅ **Section 4**: Snowflake content retrieval  
✅ **Section 5**: LLM rule extraction with Pydantic validation  
✅ **Section 6**: JSON export for analysis  

**Key Features**:
- No mock data dependencies (uses real Snowflake and API)
- Editable prompts for experimentation
- Company domain filtering
- High reasoning LLM for quality extraction
- Comprehensive error handling

**Note**: Make sure to run `uv sync` first to install the `secure-snowflake-connector` dependency before running this notebook.

In [23]:
results[0].keys()

dict_keys(['policy_metadata', 'results', 'PLCY_ID', 'PAYOR_NM', 'PLCY_URL'])

In [24]:
df_results = pd.DataFrame(results)
df_results.head(1)

,policy_metadata,results,PLCY_ID,PAYOR_NM,PLCY_URL
0,"{'policy_title': 'Obstetrical Services Policy,...",[{'code': '59514 (mapped from Cesarean Section...,RP_GBD_UHC_00214,United Health,https://www.uhcprovider.com/content/dam/provid...


In [25]:
df_results['PAYOR_NM'].unique()

<ArrowStringArray>
[                 'United Health',             'Premera Blue Cross',
                          'Cigna',                  'Highmark_BCBS',
                         'Molina',     'Elevance Health (external)',
              'Kaiser Permanente', 'Kaiser Foundation Health Plans']
Length: 8, dtype: str

In [26]:
system_prompt_policy_cpt_process = """
You are an expert at analyzing insurance policies and extracting key information.
Read the policy text from the user and return valid JSON only.
Follow the exact schema provided.
Do not output markdown or explanatory text.
Do not hallucinate.
Use null or [] when information is missing.
"""

user_prompt_aggregate_at_payor_level = """You are aggregating policy findings at the PAYOR level.
CPT Codes:
```
{cpt_codes}
```

Payor policies for processing :
```
{policies}
```

Task:
Review all provided policies for this payor and aggregate the findings at the payor level.

Rules:
- Only use the policy content provided above.
- Produce findings separately for each requested CPT/HCPCS code.
- Aggregate across all policies for the payor.
- If multiple policies agree, consolidate into a single summarized finding.
- If policies differ, do NOT force a single conclusion. Capture the variation clearly in the relevant field.
- If a finding applies only in certain situations (for example by provider type, site of service, claim type, diagnosis, modifier, place of service, or code combination), include those conditions explicitly.
- If the policies do not directly mention a requested code, return that code with empty fields.
- Do not infer unsupported rules. Do not use outside knowledge.
- Keep each field concise but complete.
- Return valid JSON only.

Output JSON schema:
```json
{{
  "results": [
    {{
      "code": "string",
      "mention_status": "mentioned | not_mentioned"
      "payor_level_summary": "string",
      "site_of_service": "string",
      "bundling_logic": "string",
      "code_interactions": "string",
      "modifier_usage": "string",
      "denial_conditions": "string",
      "unit_pricing_logic": "string",
      "documentation_requirements": "string",
      "policy_variations": [
        {{
          "policy_name": "string",
          "dimension": "site_of_service | bundling_logic | code_interactions | modifier_usage | denial_conditions | unit_pricing_logic | documentation_requirements | other",
          "variation": "string"
        }}
      ],
      "source_policies": [
        "string"
      ]
    }}
  ]
}}
```
Field guidance:
- code: CPT/HCPCS code being analyzed.
- payor_level_summary: Overall aggregated summary for this code across all provided policies.
- site_of_service: Allowed, restricted, excluded, or conditional places/settings/provider billing contexts.
- bundling_logic: Whether the code is bundled, inclusive, mutually exclusive, or not separately reimbursed.
- code_interactions: Interactions with other codes, including same-day limits, precedence, replacement logic, or incompatibilities.
- modifier_usage: Allowed, required, disallowed, or conditionally accepted modifiers.
- denial_conditions: Situations where the claim may be denied based on the provided policies.
- unit_pricing_logic: Time rules, unit limits, frequency limits, quantity rules, and reimbursement unit logic.
- documentation_requirements: Documentation needed to support reimbursement or avoid denial.
- policy_variations: Only include when policies differ materially. If no meaningful variation exists, return an empty array.
- source_policies: List the policy names/titles used for this code, if available from the input. Otherwise return an empty array.
"""

In [27]:
payor_level_summary = []

for payor in df_results['PAYOR_NM'].unique():
    print(f"Processing payor: {payor}")
    grp = df_results[df_results['PAYOR_NM'] == payor]
    try:
        messages = [
            {"role": "system", "content": system_prompt_policy_cpt_process},
            {"role": "user", "content": user_prompt_aggregate_at_payor_level.format(policies=grp['results'].tolist(), cpt_codes=cpt_codes)}
        ]

        llm_response = llm.invoke(messages)
        content = llm_response.content.strip()
        
        # Remove markdown code fences if present
        if content.startswith("```"):
            lines = content.split("\\n")
            if len(lines) > 2:
                content = "\\n".join(lines[1:-1])
        
        # Parse JSON
        llm_response = json.loads(content)
        llm_response["payor"] = payor
        payor_level_summary.append(llm_response.copy())
    except Exception as e:
        print(f"Error processing policy: {e}")
        # try:
        #     print("Retrying .....")
        #     time.sleep(30)
        #     llm_response = ehap_base_api_call(system_prompt_policy_cpt_process, 
        #                           user_prompt_aggregate_at_payor_level.format(policies=grp.ai_summary_of_policy_cpt.tolist(), cpt_codes=top_procedure_code),
        #                           EHAP, {'qos': 'accurate', 'preview': 'true', 'reasoning': 'true'})
        #     llm_response["payor"] = payor
        #     payor_level_summary.append(llm_response)
        # except Exception as e2:
        #     print(f"Error again processing policy : {e2}")
        #     payor_level_summary.append(None)
        # continue

Processing payor: United Health
Processing payor: Premera Blue Cross
Processing payor: Cigna
Processing payor: Highmark_BCBS
Processing payor: Molina
Processing payor: Elevance Health (external)
Processing payor: Kaiser Permanente
Processing payor: Kaiser Foundation Health Plans


In [28]:
# Export results to JSON
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"reimbursement_agent_payor_smry_{timestamp}.json"

with open(output_file, 'w') as f:
    json.dump(payor_level_summary, f, indent=2)

print(f"[EXPORT] ✓ Results saved to: {output_file}")


[EXPORT] ✓ Results saved to: reimbursement_agent_payor_smry_20260527_151538.json


## Section 7: Node 4 - Analyze Table Structure

This section analyzes the extracted policy results to determine the optimal table structure for the summary output.

### 7.1 Collect Rule Categories

Collect all non-empty rule categories across all extracted policies to determine which categories have meaningful data.

In [29]:
# Collect all non-empty rule categories across all policies
print("[TABLE STRUCTURE] Collecting rule categories from extracted policies...")
print("="*80)

category_samples = {
    "site_of_service": [],
    "bundling_logic": [],
    "code_interactions": [],
    "modifier_usage": [],
    "denial_conditions": [],
    "unit_pricing_logic": [],
    "documentation_requirements": [],
    "evidence_summary": []
}

for policy_result in results:
    if policy_result is None:
        continue
    
    policy_rules = policy_result.get("results", [])
    for rule in policy_rules:
        for category in category_samples.keys():
            value = (rule.get(category) or "").strip()
            if value and value not in category_samples[category]:
                category_samples[category].append(value)

# Filter out empty categories
category_samples = {k: v for k, v in category_samples.items() if v}

print(f"[TABLE STRUCTURE] Found {len(category_samples)} non-empty categories:")
for cat, samples in category_samples.items():
    print(f"  - {cat}: {len(samples)} unique value(s)")
    if samples:
        print(f"    Sample: {samples[0][:80]}...")
print("="*80)

[TABLE STRUCTURE] Collecting rule categories from extracted policies...
[TABLE STRUCTURE] Found 8 non-empty categories:
  - site_of_service: 49 unique value(s)
    Sample: CMS-1500 professional claims only; hospital and home/non-facility deliveries are...
  - bundling_logic: 57 unique value(s)
    Sample: Use the most appropriate delivery-only CPT code in Louisiana. Cesarean delivery ...
  - code_interactions: 63 unique value(s)
    Sample: CC/MCC and sterilization status are not used in this professional CPT policy. Lo...
  - modifier_usage: 48 unique value(s)
    Sample: Modifier 22 may be used for increased procedural services and multiple gestation...
  - denial_conditions: 61 unique value(s)
    Sample: Denial or rejection may occur if a global or delivery-plus-postpartum cesarean c...
  - unit_pricing_logic: 50 unique value(s)
    Sample: Report one unit for the delivery service. Louisiana modifier 22 pricing is 125% ...
  - documentation_requirements: 53 unique value(s)
    Samp

### 7.2 Use All Categories

Use all non-empty categories individually for the summary table (no combining or selection).

In [30]:
# Use ALL categories individually (no combining or selection)
print("[TABLE STRUCTURE] Using all categories individually...")
print("="*80)

# Create individual category entries for all non-empty categories
selected_categories = []
for category_name in category_samples.keys():
    selected_categories.append({
        "id": category_name,
        "categories": [category_name]  # Single category, not combined
    })

print(f"[TABLE STRUCTURE] Using {len(selected_categories)} individual categories:")
for cat in selected_categories:
    print(f"  - {cat['id']}: {len(category_samples[cat['id']])} unique values")
print("="*80)

[TABLE STRUCTURE] Using all categories individually...
[TABLE STRUCTURE] Using 8 individual categories:
  - site_of_service: 49 unique values
  - bundling_logic: 57 unique values
  - code_interactions: 63 unique values
  - modifier_usage: 48 unique values
  - denial_conditions: 61 unique values
  - unit_pricing_logic: 50 unique values
  - documentation_requirements: 53 unique values
  - evidence_summary: 70 unique values


### 7.3 Generate Column Labels with LLM (Editable Prompt)

Use LLM to generate concise, professional column labels for the selected category combinations.

In [31]:
# ============= COLUMN LABEL GENERATION PROMPT (EDIT HERE) =============

# Build samples for LLM
samples_text = ""
for cat_combo in selected_categories:
    cat_id = cat_combo["id"]
    categories = cat_combo["categories"]
    category_name = categories[0]  # Single category
    
    if category_name in category_samples and category_samples[category_name]:
        sample_text = category_samples[category_name][0][:80]
        samples_text += f"\n{cat_id}:\n  Sample: {sample_text}...\n"

COLUMN_LABEL_PROMPT = f"""Given these policy rule categories and sample content:
{samples_text}

Generate concise column labels (max 5 words each) for a summary table.
Labels should be clear, professional, and suitable for table headers.

IMPORTANT: Output ONLY valid JSON, no markdown code fences.

Output JSON:
{{
  "columns": [
    {{"id": "category_id", "label": "Column Label (max 5 words)", "type": "text"}}
  ]
}}
"""

print("[LLM] Generating column labels for all categories...")
print("="*80)

if SHOW_FULL_PROMPTS:
    print("\n[PROMPT] Column Label Generation:")
    print("-"*80)
    print(COLUMN_LABEL_PROMPT)
    print("-"*80)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": COLUMN_LABEL_PROMPT}
]

try:
    response = llm.invoke(messages)
    content = response.content.strip()
    
    if SHOW_RAW_RESPONSES:
        print("\n[RESPONSE] LLM Response:")
        print("-"*80)
        print(content)
        print("-"*80)
    
    # Remove markdown code fences if present
    if content.startswith("```"):
        lines = content.split("\n")
        if len(lines) > 2:
            content = "\n".join(lines[1:-1])
    
    # Parse JSON
    data = json.loads(content)
    
    # Validate with Pydantic
    validated = ColumnLabelsResponse(**data)
    column_metadata = [col.model_dump() for col in validated.columns]
    
    print(f"\n[SUCCESS] Generated {len(column_metadata)} column label(s)")
    for col in column_metadata:
        print(f"  - {col['id']}: '{col['label']}' ({col['type']})")
    
except Exception as e:
    print(f"\n[ERROR] {e}")
    print("[FALLBACK] Using simple column labels")
    
    # Fallback to simple labels
    column_metadata = [
        {
            "id": cat["id"],
            "label": cat["id"].replace("_", " ").title(),
            "type": "text"
        }
        for cat in selected_categories
    ]
    
    for col in column_metadata:
        print(f"  - {col['id']}: '{col['label']}' ({col['type']})")

print("="*80)

[LLM] Generating column labels for all categories...

[PROMPT] Column Label Generation:
--------------------------------------------------------------------------------
Given these policy rule categories and sample content:

site_of_service:
  Sample: CMS-1500 professional claims only; hospital and home/non-facility deliveries are...

bundling_logic:
  Sample: Use the most appropriate delivery-only CPT code in Louisiana. Cesarean delivery ...

code_interactions:
  Sample: CC/MCC and sterilization status are not used in this professional CPT policy. Lo...

modifier_usage:
  Sample: Modifier 22 may be used for increased procedural services and multiple gestation...

denial_conditions:
  Sample: Denial or rejection may occur if a global or delivery-plus-postpartum cesarean c...

unit_pricing_logic:
  Sample: Report one unit for the delivery service. Louisiana modifier 22 pricing is 125% ...

documentation_requirements:
  Sample: Standard operative and delivery record. Modifier 22 generall

## Section 8: Node 5 - Format Output

This section formats the extracted rules into the final output structure with a summary table and individual policies list.

### 8.1 Build Individual Policies List

Format each policy with metadata and evidence summary for the individual policies section.

In [32]:
# Build individual policies list and group by payer
from collections import defaultdict
print("[FORMAT] Building individual policies list...")
print("="*80)

individual_policies = []
policies_by_payer = defaultdict(list)

for i, policy_result in enumerate(results):
    if policy_result is None:
        continue
    
    # Get metadata
    llm_metadata = policy_result.get("policy_metadata", {})
    policy_results_data = policy_result.get("results", [])
    
    # Extract payer name
    payer_name = policy_result.get("PAYOR_NM", "Unknown Payer")
    
    # Determine policy title
    policy_title = llm_metadata.get("policy_title", "Policy Document")
    
    # Determine tags from payer category
    payer_category = llm_metadata.get("payer_category", "Commercial")
    tags = [payer_category] if payer_category else []
    
    # Get effective date
    effective_date = llm_metadata.get("effective_date", "N/A")
    
    # Extract evidence summary (look in denial_conditions or evidence_summary)
    evidence = "No specific denial policy documented"
    if policy_results_data:
        first_result = policy_results_data[0]
        evidence = first_result.get("denial_conditions") or first_result.get("evidence_summary", evidence)
    
    # Get policy URL
    policy_url = policy_result.get("PLCY_URL", "")
    
    # Build policy entry
    policy_entry = {
        "payer_name": payer_name,
        "policy_title": policy_title,
        "tags": tags,
        "effective_date": effective_date,
        "evidence": evidence,
        "policy_url": policy_url,
        "results": policy_results_data,  # Will be removed later
        "metadata": llm_metadata  # Will be removed later
    }
    
    individual_policies.append(policy_entry)
    policies_by_payer[payer_name].append(policy_entry)

print(f"[FORMAT] Built {len(individual_policies)} individual policy entries")
print(f"[FORMAT] Grouped into {len(policies_by_payer)} payer(s):")
for payer, policies in policies_by_payer.items():
    print(f"  - {payer}: {len(policies)} policy(ies)")
print("="*80)

[FORMAT] Building individual policies list...
[FORMAT] Built 32 individual policy entries
[FORMAT] Grouped into 8 payer(s):
  - United Health: 13 policy(ies)
  - Premera Blue Cross: 5 policy(ies)
  - Cigna: 4 policy(ies)
  - Highmark_BCBS: 2 policy(ies)
  - Molina: 2 policy(ies)
  - Elevance Health (external): 4 policy(ies)
  - Kaiser Permanente: 1 policy(ies)
  - Kaiser Foundation Health Plans: 1 policy(ies)


### 8.2 Aggregate Rules Per Payer (Editable Prompt)

For each payer, aggregate rules across all their policies using LLM summarization.

In [33]:
# ============= RULE AGGREGATION PROMPT (EDIT HERE) =============

def aggregate_payer_rules(payer_policies, category_id, categories, payer_name):
    """
    Aggregate rules for a payer across multiple policies with Pydantic validation.
    
    Args:
        payer_policies: List of policies for this payer
        category_id: Category combination ID
        categories: List of rule categories to aggregate
        payer_name: Name of the payer
        
    Returns:
        Aggregated rule text (max 15 words)
    """
    # Collect all rule texts for these categories
    rule_texts = []
    for policy in payer_policies:
        policy_results_data = policy.get("results", [])
        for result in policy_results_data:
            for cat in categories:
                text = (result.get(cat, "") or "").strip()
                if text and text not in rule_texts:
                    rule_texts.append(text)
    
    if not rule_texts:
        return "-"
    
    if len(rule_texts) == 1:
        return rule_texts[0]
    
    # Multiple rules - use LLM to summarize
    rules_list = "\n".join([f"- {text}" for text in rule_texts[:5]])  # Max 5 rules
    
    RULE_AGGREGATION_PROMPT = f"""Summarize these policy rules for {payer_name} into ONE concise statement (max 15 words):

{rules_list}

IMPORTANT: Output only the summary text. No JSON, no markdown, just the plain text summary."""
    
    messages = [
        {"role": "system", "content": "You are a concise policy summarizer. Output only the requested summary text."},
        {"role": "user", "content": RULE_AGGREGATION_PROMPT}
    ]
    
    try:
        response = llm.invoke(messages)
        summary_text = response.content.strip()
        
        # Validate with Pydantic (will auto-truncate if needed)
        validated = RuleSummary(summary=summary_text)
        
        return validated.summary
        
    except ValidationError as e:
        print(f"  [WARNING] Validation error for {payer_name} - {category_id}: {e}")
        # Fallback: return first rule (may exceed word limit but at least has content)
        return rule_texts[0]
    except Exception as e:
        print(f"  [WARNING] Failed to summarize rules for {payer_name} - {category_id}: {e}")
        # Fallback: return first rule
        return rule_texts[0]

print("[FORMAT] Aggregating rules per payer for each category...")
print("="*80)

# Pre-compute aggregated rules for each payer
payer_aggregated_rules = {}

for payer_name, payer_policies in policies_by_payer.items():
    print(f"\n[PAYER] {payer_name}")
    payer_aggregated_rules[payer_name] = {}
    
    for cat_combo in selected_categories:
        category_id = cat_combo["id"]
        categories = cat_combo["categories"]
        
        print(f"  Aggregating {category_id}...")
        aggregated_text = aggregate_payer_rules(payer_policies, category_id, categories, payer_name)
        payer_aggregated_rules[payer_name][category_id] = aggregated_text
        print(f"    Result: {aggregated_text[:80]}...")

print("\n" + "="*80)
print("[FORMAT] ✓ Rule aggregation complete")

[FORMAT] Aggregating rules per payer for each category...

[PAYER] United Health
  Aggregating site_of_service...
    Result: United Health covers CMS-1500 obstetrical professional claims only, including Me...
  Aggregating bundling_logic...
    Result: Louisiana obstetric billing requires appropriate delivery/global CPTs, bundling ...
  Aggregating code_interactions...
    Result: United Health Louisiana: use delivery-only OB coding; separate anesthesia sessio...
  Aggregating modifier_usage...
    Result: Use required anesthesia and applicable OB modifiers; 22 for increased/multiple g...
  Aggregating denial_conditions...
    Result: United Health may deny Louisiana OB/anesthesia claims for improper coding, unbun...
  Aggregating unit_pricing_logic...
    Result: Delivery/global OB one unit; anesthesia minute-based; modifier 22/AD special pri...
  Aggregating documentation_requirements...
    Result: Usually submit records for modifier 22/itemized OB care, report anesthesia minut...


### 8.3 Build Summary Table

Construct the final summary table with dynamic columns and one row per payer.

In [34]:
# Build summary table structure
print("[FORMAT] Building summary table...")
print("="*80)

# Build columns
columns = [
    {"id": "payer_org", "label": "Payer Organization", "type": "text"}
]

# Add dynamic rule columns
for col_meta in column_metadata:
    columns.append(col_meta)

# Add effective date column
columns.append({
    "id": "policy_effective_date",
    "label": "Policy Effective Date",
    "type": "date"
})

print(f"[COLUMNS] Table will have {len(columns)} columns:")
for col in columns:
    print(f"  - {col['id']}: '{col['label']}' ({col['type']})")

# Build rows
summary_rows = []

for payer_name, payer_policies in policies_by_payer.items():
    print(f"\n[ROW] Building row for {payer_name}")
    
    row = {"payer_org": payer_name}
    
    # Get most recent effective date
    dates = []
    for policy in payer_policies:
        date_str = policy.get("effective_date", "")
        if date_str and date_str != "N/A":
            try:
                # Try to parse date for comparison
                dates.append((date_str, datetime.strptime(date_str, "%m/%d/%Y")))
            except:
                dates.append((date_str, None))
    
    if dates:
        # Sort by parsed date if available
        dates_sorted = sorted([d for d in dates if d[1]], key=lambda x: x[1], reverse=True)
        if dates_sorted:
            row["policy_effective_date"] = dates_sorted[0][0]
        else:
            row["policy_effective_date"] = dates[0][0]
    else:
        row["policy_effective_date"] = "N/A"
    
    # Add aggregated rule columns
    for cat_combo in selected_categories:
        category_id = cat_combo["id"]
        row[category_id] = payer_aggregated_rules[payer_name][category_id]
    
    summary_rows.append(row)
    print(f"  ✓ Row complete: {len(row)} fields")

# Build summary table structure
summary_table = {
    "title": "Payer Policy Summary",
    "subtitle": f"Policy Analysis for CPT codes {cpt_codes}",
    "columns": columns,
    "rows": summary_rows
}

print(f"\n[SUMMARY TABLE]")
print(f"  Title: {summary_table['title']}")
print(f"  Subtitle: {summary_table['subtitle']}")
print(f"  Columns: {len(columns)}")
print(f"  Rows: {len(summary_rows)}")
print("="*80)

[FORMAT] Building summary table...
[COLUMNS] Table will have 10 columns:
  - payer_org: 'Payer Organization' (text)
  - site_of_service: 'Site of Service' (text)
  - bundling_logic: 'Bundling Rules' (text)
  - code_interactions: 'Code Interactions' (text)
  - modifier_usage: 'Modifier Usage' (text)
  - denial_conditions: 'Denial Conditions' (text)
  - unit_pricing_logic: 'Unit Pricing Rules' (text)
  - documentation_requirements: 'Documentation Required' (text)
  - evidence_summary: 'Evidence Summary' (text)
  - policy_effective_date: 'Policy Effective Date' (date)

[ROW] Building row for United Health
  ✓ Row complete: 10 fields

[ROW] Building row for Premera Blue Cross
  ✓ Row complete: 10 fields

[ROW] Building row for Cigna
  ✓ Row complete: 10 fields

[ROW] Building row for Highmark_BCBS
  ✓ Row complete: 10 fields

[ROW] Building row for Molina
  ✓ Row complete: 10 fields

[ROW] Building row for Elevance Health (external)
  ✓ Row complete: 10 fields

[ROW] Building row for Kaise

### 8.4 Build Final Formatted Output

Combine the summary table and individual policies into the final output structure.

In [35]:
# Clean up individual policies (remove temporary fields)
print("[FORMAT] Finalizing output...")
print("="*80)

for policy in individual_policies:
    policy.pop("results", None)
    policy.pop("metadata", None)

# Build final formatted output
formatted_output = {
    "summary_table": summary_table,
    "individual_policies": individual_policies
}

print(f"[FORMATTED OUTPUT]")
print(f"  Summary table:")
print(f"    - {len(summary_rows)} payer rows")
print(f"    - {len(columns)} columns")
print(f"  Individual policies:")
print(f"    - {len(individual_policies)} policy entries")
print("\n" + "="*80)
print("[FORMAT] ✓ Formatted output complete")

[FORMAT] Finalizing output...
[FORMATTED OUTPUT]
  Summary table:
    - 8 payer rows
    - 10 columns
  Individual policies:
    - 32 policy entries

[FORMAT] ✓ Formatted output complete


## Section 9: Display Formatted Results

View the formatted summary table and individual policies.

### 9.1 Display Summary Table

In [36]:
# Display summary table as DataFrame
import pandas as pd

summary_df = pd.DataFrame(summary_table["rows"])

print("="*120)
print(f"  {summary_table['title']}")
print(f"  {summary_table['subtitle']}")
print("="*120)
display(summary_df)
print("="*120)

  Payer Policy Summary
  Policy Analysis for CPT codes ["Cesarean Section without Sterilization without CC/MCC", "Vaginal Delivery without Sterilization/D&C with CC", "Vaginal Delivery Without Sterilization or D&C without CC/MCC"]


,payer_org,policy_effective_date,site_of_service,bundling_logic,code_interactions,modifier_usage,denial_conditions,unit_pricing_logic,documentation_requirements,evidence_summary
0,United Health,05/01/2026,United Health covers CMS-1500 obstetrical prof...,Louisiana obstetric billing requires appropria...,United Health Louisiana: use delivery-only OB ...,Use required anesthesia and applicable OB modi...,United Health may deny Louisiana OB/anesthesia...,Delivery/global OB one unit; anesthesia minute...,Usually submit records for modifier 22/itemize...,United Health maps delivery-only OB cases to 0...
1,Premera Blue Cross,10/07/2025,Applies mainly to professional maternity claim...,Premera maternity billing: single cesarean glo...,Global maternity bundles exclude TH E/M; modif...,TH limited to antepartum visits; 52 restricted...,Premera may deny/reduce reimbursement for unsu...,Non-time-based deliveries: separate birth line...,Document TH and reduced/discontinued services;...,Professional maternity policy: antepartum-only...
2,Cigna,04/01/2011,Professional CMS-1500 claims only; inpatient D...,Use appropriate delivery CPTs; modifier 22/DRG...,Modifier 22 not for E&M; multiple-birth cesare...,Use modifier 22 for unusually complex cesarean...,"Denials apply for unsupported modifier 22, mis...","Cigna reimburses primary deliveries 100%, addi...",Submit detailed obstetric and modifier-22 docu...,Cigna uses general modifier 22; cesarean multi...
3,Highmark_BCBS,03/12/2018,Commercial inpatient obstetrical delivery poli...,Obstetric anesthesia add-ons require primary c...,"Apply delivery-date, add-on, and code-combinat...",Use correct anesthesia and obstetrical modifie...,Require matching primary 01967; deny separate ...,Anesthesia pays base plus timed units; obstetr...,Submit records supporting anesthesia times/att...,No direct DRG rules; delivery policies bundle ...
4,Molina,10/23/2023,"Same provider, same day, single operative sess...","Molina applies general MPPR: primary 100%, sec...",Same-day vaginal delivery with cesarean isn't ...,-,Missing indicators/documentation or Medicaid/C...,No time-based unit rule stated. Payment follow...,Submit all necessary indicators and supporting...,"Molina CA Medi-Cal applies MPPR, denies same-d..."
5,Elevance Health (external),07/17/2024,Professional claims only for same- or combined...,Multiple births reimbursable; only one cesarea...,Multiple-birth extra cesareans aren’t separate...,Multiple births: modifier 22 for cesarean deli...,"Claims denied for improper obstetric coding, m...",Elevance reimburses one Cesarean per multiple ...,Documentation must support billed delivery ser...,"For multiple births, one cesarean reimbursed; ..."
6,Kaiser Permanente,09/01/2014,Inpatient hospital obstetric delivery is restr...,Early elective deliveries and related anesthes...,Denials depend on elective timing and medical ...,-,Denied for elective early-term delivery betwee...,Bill according to CMS guidelines; no separate ...,"Require gestational age, medical indication, a...",No reimbursement for elective 37–39-week deliv...
7,Kaiser Foundation Health Plans,09/01/2014,Kaiser denies nonmedically indicated elective ...,"If denied as an early elective delivery, relat...",Medical necessity governs elective early deliv...,-,Non-medically indicated elective or cesarean d...,-,Documentation of gestational age and medical i...,Elective 37-39 week deliveries without medical...


### 9.2 Display Individual Policies

In [37]:
# Display individual policies
print("="*120)
print(f"  Individual Policies ({len(individual_policies)} total)")
print("="*120)

for i, policy in enumerate(individual_policies[:10], 1):  # Show first 10
    print(f"\n{i}. {policy['payer_name']} - {policy['policy_title']}")
    print(f"   Tags: {', '.join(policy['tags'])}")
    print(f"   Effective Date: {policy['effective_date']}")
    print(f"   Evidence: {policy['evidence'][:150]}...")
    print(f"   URL: {policy['policy_url'][:80]}...")

if len(individual_policies) > 10:
    print(f"\n... and {len(individual_policies) - 10} more policies")

print("\n" + "="*120)

  Individual Policies (32 total)

1. United Health - Obstetrical Services Policy, Professional for Louisiana
   Tags: Medicaid
   Effective Date: 02/02/2024
   Evidence: Denial or rejection may occur if a global or delivery-plus-postpartum cesarean code is billed instead of delivery-only coding in Louisiana, included s...
   URL: https://www.uhcprovider.com/content/dam/provider/docs/public/policies/medicaid-c...

2. United Health - Anesthesia Policy, Professional for Louisiana
   Tags: Medicaid
   Effective Date: 10/20/2025
   Evidence: Missing required anesthesia modifier; duplicate same code on the same date without a distinct-session modifier; anesthesia billed with the crosswalk c...
   URL: https://www.uhcprovider.com/content/dam/provider/docs/public/policies/medicaid-c...

3. United Health - Obstetrical Policy, Professional
   Tags: Commercial
   Effective Date: 02/01/2026
   Evidence: Nonpayment is likely if billed as a DRG/facility concept under this professional policy, if bun

### 9.3 Export Formatted Output

In [38]:
# Export formatted output to JSON
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
formatted_output_file = f"reimbursement_formatted_output_{timestamp}.json"

with open(formatted_output_file, 'w', encoding='utf-8') as f:
    json.dump(formatted_output, f, indent=2, ensure_ascii=False)

print(f"[EXPORT] ✓ Formatted output saved to: {formatted_output_file}")
print(f"[EXPORT]   Summary table: {len(summary_rows)} rows x {len(columns)} columns")
print(f"[EXPORT]   Individual policies: {len(individual_policies)}")

# Also export summary table to CSV for easy viewing
summary_csv_file = f"reimbursement_summary_table_{timestamp}.csv"
summary_df.to_csv(summary_csv_file, index=False)
print(f"[EXPORT] ✓ Summary table saved to CSV: {summary_csv_file}")

[EXPORT] ✓ Formatted output saved to: reimbursement_formatted_output_20260527_151945.json
[EXPORT]   Summary table: 8 rows x 10 columns
[EXPORT]   Individual policies: 32
[EXPORT] ✓ Summary table saved to CSV: reimbursement_summary_table_20260527_151945.csv


## Section 10: Pipeline Summary

**Complete Reimbursement Agent Pipeline Executed Successfully**

✅ **Section 1**: Environment setup and configuration  
✅ **Section 2**: CPT code input and validation  
✅ **Section 3**: Policy search with LLM keyword detection  
✅ **Section 4**: Snowflake content retrieval  
✅ **Section 5**: LLM rule extraction with Pydantic validation  
✅ **Section 6**: Raw results export  
✅ **Section 7**: Table structure analysis (dynamic column selection)  
✅ **Section 8**: Output formatting (summary table + individual policies)  
✅ **Section 9**: Display and export formatted results  

**Key Features**:
- **Dynamic Table Structure**: Automatically selects most relevant rule categories
- **LLM-Generated Column Labels**: Professional, concise column headers
- **Payer-Level Aggregation**: Combines rules across multiple policies per payer
- **Pydantic Validation**: Ensures data quality with `ColumnLabelsResponse` and `RuleSummary` models
- **Editable Prompts**: All LLM prompts are editable for experimentation

**Output Files**:
- `reimbursement_agent_results_*.json` - Raw extraction results
- `reimbursement_formatted_output_*.json` - Formatted output with summary table
- `reimbursement_summary_table_*.csv` - Summary table in CSV format

## Experiment Zone

Use this section to experiment with different:
- **Extraction prompts** (Section 5.1)
- **Column label generation prompts** (Section 7.3)
- **Rule aggregation prompts** (Section 8.2)
- **Category combinations** (Section 7.2)

### Tips for Experimentation:
1. Modify prompts in the marked sections (EDIT HERE)
2. Re-run individual cells to test changes
3. Compare different outputs by exporting with different timestamps
4. Try different LLM reasoning efforts (low/medium/high)
5. Experiment with max word limits in `RuleSummary` validation

## Section 7: Elevance vs Competitors Comparison Table

This section creates a structured comparison table with Elevance Health policies compared against competitor payors.

In [ ]:
### 7.1 Define Comparison Models and Helper Functions

from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional

# Pydantic models for comparison table
class CodeSpecificDetail(BaseModel):
    """Per-code detail when policies differ by code."""
    code: str = Field(..., description="CPT/HCPCS code")
    summary: str = Field(..., description="Summary for this specific code")

class DimensionComparison(BaseModel):
    """Comparison data for a single dimension across a payer."""
    dimension_name: str = Field(..., description="Name of the policy dimension")
    combined_summary: str = Field(..., description="Aggregated summary across all codes")
    code_specific_details: List[CodeSpecificDetail] = Field(
        default_factory=list, 
        description="Per-code breakdown if policies differ by code"
    )

class PayerComparison(BaseModel):
    """Complete comparison data for a single payer."""
    payer_name: str = Field(..., description="Payer organization name")
    is_elevance: bool = Field(..., description="Whether this is Elevance/Anthem")
    dimensions: List[DimensionComparison] = Field(..., description="Policy dimensions compared")

class ComparisonTableMetadata(BaseModel):
    """Metadata about the comparison process."""
    total_payers: int = Field(..., description="Total number of payers")
    elevance_identified: bool = Field(..., description="Whether Elevance was found")
    competitors_count: int = Field(..., description="Number of competitor payers")
    timestamp: str = Field(..., description="Generation timestamp")
    dimension_selection_method: str = Field(..., description="How dimensions were selected")

class ComparisonTable(BaseModel):
    """Root comparison table structure."""
    codes_analyzed: List[str] = Field(..., description="List of codes being compared")
    dimensions_selected: List[str] = Field(..., description="Dimensions chosen for comparison")
    elevance_payer_name: Optional[str] = Field(None, description="Identified Elevance payer name")
    rows: List[PayerComparison] = Field(..., description="Payer comparison rows")
    metadata: ComparisonTableMetadata = Field(..., description="Comparison metadata")

# Helper functions
def identify_elevance_payer(payer_summaries: List[Dict]) -> Optional[str]:
    """Identify which payer is Elevance/Anthem."""
    elevance_keywords = ["elevance", "anthem"]
    
    for summary in payer_summaries:
        payer_name = summary.get("payor", "").lower()
        for keyword in elevance_keywords:
            if keyword in payer_name:
                return summary.get("payor")
    
    return None

def calculate_dimension_variance(payer_summaries: List[Dict], dimension: str) -> float:
    """Calculate variance/information content for a dimension."""
    texts = []
    
    for summary in payer_summaries:
        results = summary.get("results", [])
        for result in results:
            text = result.get(dimension, "")
            if text and text.strip():
                texts.append(text)
    
    if not texts:
        return 0.0
    
    # Calculate variance based on unique texts and total length
    unique_texts = set(texts)
    avg_length = sum(len(t) for t in texts) / len(texts)
    
    # Score: uniqueness * average length
    variance_score = len(unique_texts) / len(texts) * avg_length
    
    return variance_score

def select_top_dimensions(payer_summaries: List[Dict], top_n: int = 3) -> List[str]:
    """Select most informative dimensions based on variance."""
    # Available dimensions
    available_dimensions = [
        "denial_conditions",
        "bundling_logic", 
        "unit_pricing_logic",
        "site_of_service",
        "modifier_usage",
        "code_interactions",
        "documentation_requirements"
    ]
    
    # Prioritize key dimensions
    priority_weights = {
        "denial_conditions": 1.5,
        "bundling_logic": 1.4,
        "unit_pricing_logic": 1.3,
        "site_of_service": 1.2,
        "modifier_usage": 1.1,
        "code_interactions": 1.1,
        "documentation_requirements": 1.0
    }
    
    # Calculate weighted variance scores
    dimension_scores = {}
    for dim in available_dimensions:
        variance = calculate_dimension_variance(payer_summaries, dim)
        weight = priority_weights.get(dim, 1.0)
        dimension_scores[dim] = variance * weight
    
    # Sort by score and select top N
    sorted_dims = sorted(dimension_scores.items(), key=lambda x: x[1], reverse=True)
    selected = [dim for dim, score in sorted_dims[:top_n] if score > 0]
    
    return selected

print("✓ Comparison models and helper functions defined")

In [ ]:
### 7.2 Generate Comparison Table

print("[COMPARISON] Generating Elevance vs Competitors comparison table...")
print("="*80)

# Load payor summary data
print(f"[DATA] Loaded {len(payor_level_summary)} payer summaries")

# Extract all unique codes from the summaries
all_codes = set()
for summary in payor_level_summary:
    for result in summary.get("results", []):
        code = result.get("code", "")
        if code:
            all_codes.add(code)

codes_analyzed = sorted(list(all_codes))
print(f"[CODES] Analyzing {len(codes_analyzed)} code(s): {codes_analyzed[:3]}...")

# Identify Elevance payer
elevance_payer_name = identify_elevance_payer(payor_level_summary)
if elevance_payer_name:
    print(f"[ELEVANCE] ✓ Identified Elevance payer: {elevance_payer_name}")
else:
    print("[ELEVANCE] ⚠ Warning: Could not identify Elevance payer")

# Select top dimensions for comparison
dimensions_selected = select_top_dimensions(payor_level_summary, top_n=3)
print(f"[DIMENSIONS] Selected top {len(dimensions_selected)} dimensions:")
for i, dim in enumerate(dimensions_selected, 1):
    print(f"  {i}. {dim}")

# LLM aggregation prompt template
aggregation_prompt_template = """You are aggregating policy findings across multiple codes for a SINGLE dimension.

Payer: {payer_name}
Dimension: {dimension_name}

Code-specific summaries:
{code_summaries}

Task:
Create a combined summary that captures the overall policy for this dimension across all codes.

Rules:
- If policies are similar across codes, create a single unified summary (50-100 words)
- If policies differ significantly by code, note the key differences
- Be concise but complete
- Do not hallucinate or add information not present in the summaries
- Return ONLY the summary text, no JSON, no markdown

Summary:"""

# Build comparison rows
comparison_rows = []

for summary in payor_level_summary:
    payer_name = summary.get("payor", "Unknown")
    is_elevance = (payer_name == elevance_payer_name)
    
    print(f"\n[PAYER] Processing: {payer_name} {'(ELEVANCE)' if is_elevance else ''}")
    
    dimensions = []
    
    for dimension_name in dimensions_selected:
        print(f"  - Aggregating dimension: {dimension_name}")
        
        # Collect code-specific summaries for this dimension
        code_summaries = []
        for result in summary.get("results", []):
            code = result.get("code", "")
            dim_text = result.get(dimension_name, "")
            if code and dim_text and dim_text.strip():
                code_summaries.append(f"Code: {code}\nSummary: {dim_text}")
        
        if not code_summaries:
            # No data for this dimension
            combined_summary = "No policy information available for this dimension."
            code_specific_details = []
        else:
            # Use LLM to aggregate across codes
            code_summaries_text = "\n\n".join(code_summaries)
            prompt = aggregation_prompt_template.format(
                payer_name=payer_name,
                dimension_name=dimension_name,
                code_summaries=code_summaries_text
            )
            
            messages = [
                {"role": "system", "content": "You are a concise policy aggregator. Output only the requested summary."},
                {"role": "user", "content": prompt}
            ]
            
            try:
                response = llm.invoke(messages)
                combined_summary = response.content.strip()
                
                # Check if codes have significantly different policies
                # For now, keep code_specific_details empty unless there's high variance
                code_specific_details = []
                
            except Exception as e:
                print(f"    ⚠ LLM aggregation failed: {e}")
                # Fallback: concatenate summaries
                combined_summary = " | ".join([cs.split("Summary: ")[1] for cs in code_summaries if "Summary: " in cs])
                code_specific_details = []
        
        dimension_comparison = DimensionComparison(
            dimension_name=dimension_name,
            combined_summary=combined_summary,
            code_specific_details=code_specific_details
        )
        dimensions.append(dimension_comparison)
    
    payer_comparison = PayerComparison(
        payer_name=payer_name,
        is_elevance=is_elevance,
        dimensions=dimensions
    )
    comparison_rows.append(payer_comparison)

# Sort rows: Elevance first, then others
comparison_rows.sort(key=lambda x: (not x.is_elevance, x.payer_name))

# Build metadata
metadata = ComparisonTableMetadata(
    total_payers=len(comparison_rows),
    elevance_identified=elevance_payer_name is not None,
    competitors_count=len(comparison_rows) - (1 if elevance_payer_name else 0),
    timestamp=datetime.now().isoformat(),
    dimension_selection_method="variance_based_with_priority"
)

# Create final comparison table
comparison_table = ComparisonTable(
    codes_analyzed=codes_analyzed,
    dimensions_selected=dimensions_selected,
    elevance_payer_name=elevance_payer_name,
    rows=comparison_rows,
    metadata=metadata
)

print(f"\n{'='*80}")
print(f"[COMPARISON] ✓ Comparison table generated successfully")
print(f"[SUMMARY] {metadata.total_payers} payers, {len(dimensions_selected)} dimensions, {len(codes_analyzed)} codes")
print(f"[ELEVANCE] Identified: {metadata.elevance_identified}")

In [ ]:
### 7.3 Export Comparison Table and Analysis

print("[EXPORT] Exporting comparison table...")
print("="*80)

# Generate timestamp-based filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
comparison_output_file = f"elevance_comparison_table_{timestamp}.json"

# Export to JSON with Pydantic validation
try:
    comparison_json = comparison_table.model_dump()
    
    with open(comparison_output_file, 'w') as f:
        json.dump(comparison_json, f, indent=2)
    
    print(f"[EXPORT] ✓ Comparison table saved to: {comparison_output_file}")
    print(f"[FILE SIZE] {Path(comparison_output_file).stat().st_size:,} bytes")
except Exception as e:
    print(f"[EXPORT] ✗ Error exporting: {e}")

# Create a simplified pandas view for visualization
print("\n[VISUALIZATION] Creating pandas DataFrame view...")

comparison_data = []
for row in comparison_table.rows:
    row_data = {
        "Payer": row.payer_name,
        "Is_Elevance": "✓" if row.is_elevance else ""
    }
    
    # Add dimension columns
    for dim_comp in row.dimensions:
        # Truncate long summaries for display
        summary = dim_comp.combined_summary
        if len(summary) > 100:
            summary = summary[:97] + "..."
        row_data[dim_comp.dimension_name] = summary
    
    comparison_data.append(row_data)

df_comparison = pd.DataFrame(comparison_data)

print("\n[TABLE] Comparison Table Preview:")
print("="*80)
print(df_comparison.to_string(index=False))

# Export DataFrame to CSV
csv_output_file = f"elevance_comparison_table_{timestamp}.csv"
df_comparison.to_csv(csv_output_file, index=False)
print(f"\n[CSV] ✓ DataFrame exported to: {csv_output_file}")

# Generate summary statistics
print("\n[ANALYSIS] Summary Statistics:")
print("="*80)
print(f"Total Payers: {comparison_table.metadata.total_payers}")
print(f"Elevance Identified: {'Yes' if comparison_table.metadata.elevance_identified else 'No'}")
if comparison_table.metadata.elevance_identified:
    print(f"Elevance Payer Name: {comparison_table.elevance_payer_name}")
print(f"Competitor Count: {comparison_table.metadata.competitors_count}")
print(f"\nCodes Analyzed: {len(comparison_table.codes_analyzed)}")
for i, code in enumerate(comparison_table.codes_analyzed, 1):
    print(f"  {i}. {code}")
print(f"\nDimensions Selected: {len(comparison_table.dimensions_selected)}")
for i, dim in enumerate(comparison_table.dimensions_selected, 1):
    print(f"  {i}. {dim}")

# Optional: Highlight key differences from Elevance
if comparison_table.elevance_payer_name:
    print("\n[DIFFERENCES] Analyzing Elevance vs Competitors...")
    
    elevance_row = next((r for r in comparison_table.rows if r.is_elevance), None)
    if elevance_row:
        print(f"\nElevance Policy Summary ({comparison_table.elevance_payer_name}):")
        for dim_comp in elevance_row.dimensions:
            print(f"\n  {dim_comp.dimension_name}:")
            print(f"    {dim_comp.combined_summary[:200]}...")

print("\n" + "="*80)
print("[COMPLETE] ✓ Elevance comparison analysis complete!")
print(f"[OUTPUT] JSON: {comparison_output_file}")
print(f"[OUTPUT] CSV: {csv_output_file}")

## Comparison Table Complete ✓

**Elevance vs Competitors Comparison Successfully Generated**

The comparison table provides:
- ✅ **Structured JSON output** following Pydantic models from `reimbursement_models.py`
- ✅ **Elevance-first ordering** for easy identification
- ✅ **Top 2-3 most informative dimensions** selected automatically
- ✅ **Combined summaries** aggregated across all codes using LLM
- ✅ **Exportable formats**: JSON and CSV
- ✅ **Metadata tracking**: Timestamp, method, counts

**Output Files:**
1. `elevance_comparison_table_TIMESTAMP.json` - Full structured comparison
2. `elevance_comparison_table_TIMESTAMP.csv` - Simplified table view

**Next Steps:**
- Review the JSON output for structured comparison data
- Use the CSV for quick visual comparison
- Integrate with downstream analysis or UI components

In [84]:
system_prompt_payor_recommendation = """
You are a healthcare reimbursement policy strategy analyst.

Your job is to evaluate aggregated payor-level policy findings for a CPT/HCPCS code and identify evidence-based policy optimization opportunities for Elevance Health.

Your objective:
Compare Elevance Health against peer payors and identify policy changes that Elevance could consider to potentially reduce processing cost, reduce inappropriate reimbursement, strengthen claim editing, tighten utilization controls, or improve policy clarity.

You must follow these rules:
1. Use only the information provided in the input.
2. Do not use outside knowledge.
3. Do not assume a stricter policy is automatically better.
4. Recommend a change only when the peer evidence suggests a plausible cost-control or reimbursement-control opportunity.
5. Prefer peer patterns supported by multiple payors over one-off outliers.
6. Separate:
   - observed peer differences,
   - Elevance gaps,
   - recommendations,
   - risks/tradeoffs.
7. Preserve nuance and conditions. If a rule applies only in certain settings, provider types, modifiers, claim types, diagnoses, or code combinations, state that clearly.
8. If evidence is weak, incomplete, or mixed, say so explicitly.
9. Keep the output concise, executive-friendly, and practical.
10. Do not overstate certainty.

Peer selection rules:
- Select only the top K most relevant peer payors.
- Select peers based on:
  a. strength of meaningful differences vs Elevance,
  b. relevance to cost/reimbursement control,
  c. clarity/completeness of their aggregate findings,
  d. usefulness in supporting recommendations.
- Avoid redundant peers unless repetition strengthens the evidence.

Recommendation prioritization rules:
Prioritize recommendations that are:
1. likely to reduce inappropriate reimbursement or processing cost,
2. supported by more than one peer when possible,
3. practical to operationalize,
4. understandable at an executive level.

Focus especially on these policy levers:
- site of service restrictions
- bundling logic
- code interaction rules
- modifier controls
- denial conditions
- unit/time/frequency limits
- documentation requirements

Return valid JSON only.
"""

user_prompt_payor_recommendation = """
Analyze the aggregated payor-level policy findings for the CPT/HCPCS code below and identify policy optimization opportunities for Elevance Health.

Target payor:
Elevance Health

CPT Codes:
{cpt_codes}

Business objective:
Recommend what Elevance Health could consider changing in its policy to potentially reduce processing cost, reduce inappropriate reimbursement, improve claim editing consistency, or tighten utilization management by comparing against peer payors.

Instructions:
- Use only the aggregated payor-level findings provided below.
- Compare Elevance Health against peer payors.
- Select only the top most relevant peer payors for the comparison.
- Focus on meaningful differences that may represent cost-control or reimbursement-control opportunities:
  - preventing duplicate reimbursement
  - narrowing reimbursable settings
  - tightening billing combinations
  - limiting bypass modifiers
  - restricting excessive units or time billing
  - increasing documentation thresholds
- Prefer recommendations supported by multiple peer payors over one-off outliers.
- Keep the explanation easy to understand at an executive level.
- Be concise, practical, and evidence-based.
- If the evidence is mixed, weak, or incomplete, say so clearly.
- Do not force a recommendation if the peer evidence is not strong enough.

Aggregated payor-level findings for all payors:
```json
{all_payor_aggregates}
```

Return valid JSON only using this schema:
```json
{{
  "target_payor": "Elevance Health",
  "code": "string",
  "selected_peer_payors": [
    {{
      "payor_name": "string",
      "selection_reason": "string"
    }}
  ],
  "executive_summary": [
    "string"
  ],
  "recommendations": [
    {{
      "priority_rank": 1,
      "recommendation_title": "string",
      "recommendation_type": "site_of_service | bundling_logic | code_interactions | modifier_usage | denial_conditions | unit_pricing_logic | documentation_requirements | other",
      "why_it_matters": "string",
      "current_elevance_position": "string",
      "peer_pattern": "string",
      "proposed_policy_change": "string",
      "supporting_payors": [
        "string"
      ],
      "expected_cost_impact": "High | Medium | Low",
      "implementation_complexity": "High | Medium | Low",
      "provider_abrasion_risk": "High | Medium | Low",
      "confidence": "High | Medium | Low"
    }}
  ],
  "peer_comparison_table": [
    {{
      "feature": "string",
      "elevance_health": "string",
      "peer_values": [
        {{
          "payor_name": "string",
          "value": "string"
        }}
      ],
      "gap_summary": "string"
    }}
  ],
  "missing_features_in_elevance": [
    {{
      "feature": "string",
      "description": "string",
      "observed_in_payors": [
        "string"
      ],
      "why_it_may_matter": "string"
    }}
  ],
  "risks_and_tradeoffs": [
    "string"
  ],
  "evidence_notes": [
    "string"
  ]
}}
```
"""